Note: in this notebook, df1=transactions, df2=customers (reversed from Week 1).

In [1]:
import pandas as pd

df1 = pd.read_excel('../data/raw/FinTrust_Transaction_Data.xlsx')
df2 = pd.read_excel('../data/raw/FinTrust_Customer_Data.xlsx')

In [2]:

print(df1.head())
print(df1.info())

  Transaction_ID Customer_ID Transaction_DateTime Transaction_Type  \
0     FT-T000001   FT-C01135  2026-01-01 00:00:00    Card Purchase   
1     FT-T000002   FT-C00428  2026-01-01 00:10:00  Cash Withdrawal   
2     FT-T000003   FT-C00469  2026-01-01 00:21:00         Transfer   
3     FT-T000004   FT-C01015  2026-01-01 00:32:00          Deposit   
4     FT-T000005   FT-C00870  2026-01-01 00:43:00  Cash Withdrawal   

   Amount_NGN     Channel   Device_Type Location International_Transaction  \
0     5923.80  Mobile App       Android    Enugu                        No   
1    11681.12         ATM       Android   Kaduna                        No   
2      571.91         ATM  POS Terminal     Kano                        No   
3   105367.52  Mobile App   Web Browser    Abuja                        No   
4     3049.06  Mobile App  POS Terminal    Abuja                        No   

  Transaction_Status Risk_Review_Flag  
0           Reversed              Yes  
1         Successful          

In [3]:
df1.isnull().sum()

Transaction_ID                0
Customer_ID                   0
Transaction_DateTime          0
Transaction_Type              0
Amount_NGN                    0
Channel                       0
Device_Type                  96
Location                     96
International_Transaction     0
Transaction_Status            0
Risk_Review_Flag              0
dtype: int64

In [4]:
def validate_transactions(df1):
    issues = {}
    issues['missing_device_type'] = df1['Device_Type'].isna().sum()
    issues['missing_location'] = df1['Location'].isna().sum()
    issues['duplicate_transaction_ids'] = df1['Transaction_ID'].duplicated().sum()
    issues['unexpected_amount'] = (df1['Amount_NGN'] <= 0).sum()
    issues['unexpected_risk_values'] = (~df1['Risk_Review_Flag'].isin(['Yes', 'No'])).sum()
    return issues
    

In [5]:
results = validate_transactions(df1)
print(results)

{'missing_device_type': np.int64(96), 'missing_location': np.int64(96), 'duplicate_transaction_ids': np.int64(0), 'unexpected_amount': np.int64(0), 'unexpected_risk_values': np.int64(0)}


In [6]:
both_missing = df1[df1['Device_Type'].isna() & df1['Location'].isna()]
print(len(both_missing))

4


In [7]:
missing_mask = df1['Device_Type'].isna() | df1['Location'].isna()
print(df1.groupby(missing_mask)['Risk_Review_Flag'].value_counts(normalize=True))

       Risk_Review_Flag
False  No                  0.803251
       Yes                 0.196749
True   No                  0.851064
       Yes                 0.148936
Name: proportion, dtype: float64


**Part C: Preprocessing Workflow**


In [8]:
def preprocess_transactions(df, mode='train'):
    df = df.copy()
    
    if mode == 'train':
        df = df.dropna(subset=['Device_Type', 'Location'])
    elif mode == 'inference':
        df['Device_Type'] = df['Device_Type'].fillna('Unknown')
        df['Location'] = df['Location'].fillna('Unknown')
    
    df['International_Transaction'] = df['International_Transaction'].map({'Yes': 1, 'No': 0})
    df['Risk_Review_Flag'] = df['Risk_Review_Flag'].map({'Yes': 1, 'No': 0})
    
    df['transaction_hour'] = df['Transaction_DateTime'].dt.hour
    df['transaction_dayofweek'] = df['Transaction_DateTime'].dt.dayofweek
    
    df = pd.get_dummies(df, columns=['Channel', 'Transaction_Type', 'Device_Type'], drop_first=True)
    
    return df

In [9]:
processed = preprocess_transactions(df1, mode='train')
processed.head()

,Transaction_ID,Customer_ID,Transaction_DateTime,Amount_NGN,Location,International_Transaction,Transaction_Status,Risk_Review_Flag,transaction_hour,transaction_dayofweek,...,Channel_Web,Transaction_Type_Bill Payment,Transaction_Type_Card Purchase,Transaction_Type_Cash Withdrawal,Transaction_Type_Deposit,Transaction_Type_Transfer,Device_Type_Android,Device_Type_POS Terminal,Device_Type_Web Browser,Device_Type_iOS
0,FT-T000001,FT-C01135,2026-01-01 00:00:00,5923.80,Enugu,0,Reversed,1,0,3,...,False,False,True,False,False,False,True,False,False,False
1,FT-T000002,FT-C00428,2026-01-01 00:10:00,11681.12,Kaduna,0,Successful,1,0,3,...,False,False,False,True,False,False,True,False,False,False
2,FT-T000003,FT-C00469,2026-01-01 00:21:00,571.91,Kano,0,Successful,0,0,3,...,False,False,False,False,False,True,False,True,False,False
3,FT-T000004,FT-C01015,2026-01-01 00:32:00,105367.52,Abuja,0,Successful,0,0,3,...,False,False,False,False,True,False,False,False,True,False
4,FT-T000005,FT-C00870,2026-01-01 00:43:00,3049.06,Abuja,0,Successful,0,0,3,...,False,False,False,True,False,False,False,True,False,False


In [10]:
#Checking dayofweek

import datetime
print(datetime.date(2026, 1, 1).weekday())

3


0 = Monday
3 = Thursday

**Customer Data**

**Part B: Data Validation**

In [11]:
print(df2.head())
print(df2.isnull().sum())

  Customer_ID     Customer_Name  Age  Gender           City Customer_Segment  \
0   FT-C00001   Ibrahim Adeyemi   56    Male          Lagos          Premium   
1   FT-C00002    Yusuf Mohammed   46    Male           Kano         Everyday   
2   FT-C00003     Emeka Adeyemi   32  Female  Port Harcourt         Everyday   
3   FT-C00004    David Williams   60  Female     Benin City          Premium   
4   FT-C00005  Blessing Adeyemi   25    Male         Kaduna          Premium   

  Account_Type  Tenure_Months  Digital_Engagement_Score Monthly_Income_Band  \
0      Savings             55                      84.9           500k-999k   
1      Current             79                      50.7           500k-999k   
2      Savings             43                      56.1                 1m+   
3      Premium             44                      54.0           500k-999k   
4      Savings             68                      85.1           500k-999k   

  Preferred_Channel Account_Status  
0      

In [12]:
def validate_customer(df):
    issues = {}
    
    issues['missing_values'] = df.isnull().sum().sum()
    issues['duplicate_customer_ids'] = df['Customer_ID'].duplicated().sum()
    issues['unexpected_gender'] = (~df['Gender'].isin(['Male', 'Female', 'Prefer not to say'])).sum()
    
    expected_bands = ['Below 100k', '100k-249k', '250k-499k', '500k-999k', '1m+']
    issues['unexpected_income_band'] = (~df['Monthly_Income_Band'].isin(expected_bands)).sum()
    
    return issues

In [13]:
results = validate_customer(df2)
print(results)

{'missing_values': np.int64(0), 'duplicate_customer_ids': np.int64(0), 'unexpected_gender': np.int64(0), 'unexpected_income_band': np.int64(0)}


**Part C: Preprocessing workflow**

In [14]:
print(df2['Monthly_Income_Band'].unique())

<ArrowStringArray>
['500k-999k', '1m+', 'Below 100k', '100k-249k', '250k-499k']
Length: 5, dtype: str


In [15]:
def preprocess_customers(df):
    df = df.copy()
    
    income_map = {
        'Below 100k': 0,
        '100k-249k': 1,
        '250k-499k': 2,
        '500k-999k': 3,
        '1m+': 4
    }
    df['Monthly_Income_Band'] = df['Monthly_Income_Band'].map(income_map)
    
    df = pd.get_dummies(
        df,
        columns=['Gender', 'City', 'Customer_Segment', 'Account_Type', 'Preferred_Channel', 'Account_Status'],
        drop_first=True
    )
    
    return df

In [16]:
processed_customers = preprocess_customers(df2)

In [17]:

processed_customers.head()

,Customer_ID,Customer_Name,Age,Tenure_Months,Digital_Engagement_Score,Monthly_Income_Band,Gender_Male,Gender_Prefer not to say,City_Benin City,City_Enugu,...,City_Port Harcourt,Customer_Segment_Premium,Customer_Segment_SME,Customer_Segment_Student,Account_Type_Premium,Account_Type_Savings,Preferred_Channel_USSD,Preferred_Channel_Web,Account_Status_Dormant,Account_Status_Restricted
0,FT-C00001,Ibrahim Adeyemi,56,55,84.9,3,True,False,False,False,...,False,True,False,False,False,True,False,False,False,False
1,FT-C00002,Yusuf Mohammed,46,79,50.7,3,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,FT-C00003,Emeka Adeyemi,32,43,56.1,4,False,False,False,False,...,True,False,False,False,False,True,False,False,False,False
3,FT-C00004,David Williams,60,44,54.0,3,False,False,True,False,...,False,True,False,False,True,False,False,False,True,False
4,FT-C00005,Blessing Adeyemi,25,68,85.1,3,True,False,False,False,...,False,True,False,False,False,True,False,False,False,False


**Part D:**
Data -> Validation -> Preprocessing -> Feature Preparation -> Model -> Prediction -> Output

In [18]:
#Merging Transaction_data and Customer_data via Customer_ID
merged = processed.merge(processed_customers, on='Customer_ID', how='left')

print(merged.shape)
merged.head()

(11812, 46)


,Transaction_ID,Customer_ID,Transaction_DateTime,Amount_NGN,Location,International_Transaction,Transaction_Status,Risk_Review_Flag,transaction_hour,transaction_dayofweek,...,City_Port Harcourt,Customer_Segment_Premium,Customer_Segment_SME,Customer_Segment_Student,Account_Type_Premium,Account_Type_Savings,Preferred_Channel_USSD,Preferred_Channel_Web,Account_Status_Dormant,Account_Status_Restricted
0,FT-T000001,FT-C01135,2026-01-01 00:00:00,5923.80,Enugu,0,Reversed,1,0,3,...,False,False,False,False,False,False,False,False,False,False
1,FT-T000002,FT-C00428,2026-01-01 00:10:00,11681.12,Kaduna,0,Successful,1,0,3,...,False,False,False,False,False,False,False,True,False,False
2,FT-T000003,FT-C00469,2026-01-01 00:21:00,571.91,Kano,0,Successful,0,0,3,...,False,False,False,False,False,False,False,False,False,False
3,FT-T000004,FT-C01015,2026-01-01 00:32:00,105367.52,Abuja,0,Successful,0,0,3,...,False,False,False,False,False,False,False,False,False,False
4,FT-T000005,FT-C00870,2026-01-01 00:43:00,3049.06,Abuja,0,Successful,0,0,3,...,False,False,False,False,False,True,False,False,False,False


In [19]:
print(merged.isnull().sum().sum())

0


In [20]:
model_data = merged.drop(columns=['Transaction_ID', 'Customer_ID', 'Customer_Name', 'Transaction_Status', 'Location','Transaction_DateTime'])

print(model_data.head())
print(model_data.dtypes)

   Amount_NGN  International_Transaction  Risk_Review_Flag  transaction_hour  \
0     5923.80                          0                 1                 0   
1    11681.12                          0                 1                 0   
2      571.91                          0                 0                 0   
3   105367.52                          0                 0                 0   
4     3049.06                          0                 0                 0   

   transaction_dayofweek  Channel_Mobile App  Channel_POS  Channel_USSD  \
0                      3                True        False         False   
1                      3               False        False         False   
2                      3               False        False         False   
3                      3                True        False         False   
4                      3                True        False         False   

   Channel_Web  Transaction_Type_Bill Payment  ...  City_Port Harcou

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = model_data.drop(columns=['Risk_Review_Flag'])
y = model_data['Risk_Review_Flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['Amount_NGN', 'Age', 'Tenure_Months', 'Digital_Engagement_Score', 'Monthly_Income_Band', 'transaction_hour', 'transaction_dayofweek']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [23]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.81      0.98      0.89      1908
           1       0.29      0.03      0.05       455

    accuracy                           0.80      2363
   macro avg       0.55      0.51      0.47      2363
weighted avg       0.71      0.80      0.73      2363



In [24]:

merged_sorted = merged.sort_values('Transaction_DateTime').reset_index(drop=True)

split_index = int(len(merged_sorted) * 0.8)

train_data = merged_sorted.iloc[:split_index]
test_data = merged_sorted.iloc[split_index:]

print(train_data['Transaction_DateTime'].max())  # latest date in training
print(test_data['Transaction_DateTime'].min())   # earliest date in testing

2026-03-13 23:25:00
2026-03-13 23:36:00


In [25]:
train_data = train_data.drop(columns=['Transaction_ID', 'Customer_ID', 'Customer_Name', 
                                        'Transaction_DateTime', 'Transaction_Status', 'Location'])
test_data = test_data.drop(columns=['Transaction_ID', 'Customer_ID', 'Customer_Name', 
                                      'Transaction_DateTime', 'Transaction_Status', 'Location'])

X_train_time = train_data.drop(columns=['Risk_Review_Flag'])
y_train_time = train_data['Risk_Review_Flag']
X_test_time = test_data.drop(columns=['Risk_Review_Flag'])
y_test_time = test_data['Risk_Review_Flag']

In [26]:
scaler_time = StandardScaler()
X_train_time[numeric_cols] = scaler_time.fit_transform(X_train_time[numeric_cols])
X_test_time[numeric_cols] = scaler_time.transform(X_test_time[numeric_cols])

model_time = LogisticRegression(max_iter=1000)
model_time.fit(X_train_time, y_train_time)
predictions_time = model_time.predict(X_test_time)

print(classification_report(y_test_time, predictions_time))

              precision    recall  f1-score   support

           0       0.81      0.99      0.89      1906
           1       0.45      0.04      0.07       457

    accuracy                           0.80      2363
   macro avg       0.63      0.51      0.48      2363
weighted avg       0.74      0.80      0.73      2363



**Part E: Technical Tests**

In [27]:
import numpy as np

test_results = []

#Test 1: Valid Input
valid_sample = df1.head(100).copy()
result = validate_transactions(valid_sample)
test_results.append({
    'test': 'Valid input',
    'expected': 'No issues (all zeros, or only known missing counts)',
    'actual': result,
    'status': 'PASS' if result['duplicate_transaction_ids'] == 0 and result['unexpected_amount'] == 0 else 'FAIL'
})

In [28]:
#Test 2: Missing values by deliberately introduce a null.

missing_sample = df1.head(100).copy()
missing_sample.loc[0, 'Device_Type'] = np.nan
result = validate_transactions(missing_sample)
test_results.append({
    'test': 'Missing values detected',
    'expected': 'missing_device_type >= 1',
    'actual': result['missing_device_type'],
    'status': 'PASS' if result['missing_device_type'] >= 1 else 'FAIL'
})


In [29]:
#Test 3: Unexpected category by inject an invalid Risk_Review_Flag value

bad_category_sample = df1.head(100).copy()
bad_category_sample.loc[0, 'Risk_Review_Flag'] = 'Maybe'
result = validate_transactions(bad_category_sample)
test_results.append({
    'test': 'Unexpected category detected',
    'expected': 'unexpected_risk_values >= 1',
    'actual': result['unexpected_risk_values'],
    'status': 'PASS' if result['unexpected_risk_values'] >= 1 else 'FAIL'
})


In [30]:
#Test 4: Incorrect data type; Amount_NGN as text instead of numeric

bad_type_sample = df1.head(100).copy()
bad_type_sample['Amount_NGN'] = bad_type_sample['Amount_NGN'].astype(str)
try:
    result = (bad_type_sample['Amount_NGN'] <= 0).sum()
    status = 'FAIL - should have raised a type error but did not'
except TypeError:
    status = 'PASS - function correctly failed on wrong dtype'
test_results.append({
    'test': 'Incorrect data type handling',
    'expected': 'TypeError or explicit rejection when Amount_NGN is not numeric',
    'actual': status,
    'status': status
})

In [31]:
#Test 5: Empty input; function should not crash on an empty dataframe

empty_sample = df1.iloc[0:0]
try:
    result = validate_transactions(empty_sample)
    status = f'PASS ran without crashing, returned: {result}'
except Exception as e:
    status = f'FAIL raised an error: {e}'
test_results.append({
    'test': 'Empty input handling',
    'expected': 'Function runs without crashing, returns zero counts',
    'actual': status,
    'status': status
})

for t in test_results:
    print(t)

{'test': 'Valid input', 'expected': 'No issues (all zeros, or only known missing counts)', 'actual': {'missing_device_type': np.int64(1), 'missing_location': np.int64(2), 'duplicate_transaction_ids': np.int64(0), 'unexpected_amount': np.int64(0), 'unexpected_risk_values': np.int64(0)}, 'status': 'PASS'}
{'test': 'Missing values detected', 'expected': 'missing_device_type >= 1', 'actual': np.int64(2), 'status': 'PASS'}
{'test': 'Unexpected category detected', 'expected': 'unexpected_risk_values >= 1', 'actual': np.int64(1), 'status': 'PASS'}
{'test': 'Incorrect data type handling', 'expected': 'TypeError or explicit rejection when Amount_NGN is not numeric', 'actual': 'PASS - function correctly failed on wrong dtype', 'status': 'PASS - function correctly failed on wrong dtype'}
{'test': 'Empty input handling', 'expected': 'Function runs without crashing, returns zero counts', 'actual': "PASS ran without crashing, returned: {'missing_device_type': np.int64(0), 'missing_location': np.int6

In [32]:
import pandas as pd

results_df = pd.DataFrame(test_results)
results_df

,test,expected,actual,status
0,Valid input,"No issues (all zeros, or only known missing co...","{'missing_device_type': 1, 'missing_location':...",PASS
1,Missing values detected,missing_device_type >= 1,2,PASS
2,Unexpected category detected,unexpected_risk_values >= 1,1,PASS
3,Incorrect data type handling,TypeError or explicit rejection when Amount_NG...,PASS - function correctly failed on wrong dtype,PASS - function correctly failed on wrong dtype
4,Empty input handling,"Function runs without crashing, returns zero c...","PASS ran without crashing, returned: {'missing...","PASS ran without crashing, returned: {'missing..."


In [33]:
with open('../reports/model_evaluation.txt', 'w') as f:
    f.write("=== Random Split Results ===\n")
    f.write(classification_report(y_test, predictions))
    f.write("\n=== Time-Based Split Results ===\n")
    f.write(classification_report(y_test_time, predictions_time))

results_df.to_csv('../reports/part_E_results.csv', index=False)